# GENE CRISpy — M1 Genetics Foundations

Independent study aid mapped to selected public UIC College of Medicine Block 1 objectives. Not UIC-authored or endorsed; not a complete syllabus. Synthetic examples only. No clinical interpretation, patient risk scoring, variant annotation or CRISPR design.

Source: https://medicine.uic.edu/education/md/md-curriculum/phase-1-curriculum/

Reviewed 2026-11-14. Phase 1 includes M1 and M2; this notebook covers selected Block 1 connections only.


## 1. Load the core package
CPU is sufficient. Clone the reviewed main commit and apply the tested learning update embedded in this notebook. No Firebase, API server, credentials or GPU is required.


In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path("/content/genecrispy-m1-genetics")
if not repo.exists():
    subprocess.run([
        "git", "clone", 
        "https://github.com/contrivancecompanychicago/replicatedgenecrispy2dpatientworld.git",
        str(repo)
    ], check=True)

subprocess.run(["git", "-C", str(repo), "checkout", "f6033738be3bbfbfbd602a29bc7347f38f9a7eb0"], check=True)
import json
updates = json.loads("{\"genecrispy/__init__.py\":\"\\\"\\\"\\\"GENE CRISpy Python package, version 0.2.0.\\\"\\\"\\\"\\n\\nfrom .agent import GeneCrispyAgent, MAX_INPUT_BYTES\\n\\n__version__ = \\\"0.2.0\\\"\\n\\n__all__ = [\\\"GeneCrispyAgent\\\", \\\"MAX_INPUT_BYTES\\\", \\\"__version__\\\"]\\n\\nfrom .education import get_m1_genetics_curriculum, get_genetics_lesson\\n\",\"genecrispy/agent.py\":\"\\\"\\\"\\\"Local, deterministic sequence and coordinate-file analysis for GENE CRISpy.\\n\\nThis module reports descriptive properties only. It does not interpret variants,\\nmake clinical claims, or send submitted data to an external service.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections import Counter\\nfrom typing import Any\\n\\nMAX_INPUT_BYTES = 5 * 1024 * 1024\\nDNA_ALPHABET = set(\\\"ACGTNRYSWKMBDHV\\\")\\nRNA_ALPHABET = set(\\\"ACGUNRYSWKMBDHV\\\")\\nPROTEIN_ALPHABET = set(\\\"ACDEFGHIKLMNPQRSTVWYBXZJUO*\\\")\\n\\n\\nclass GeneCrispyAgent:\\n    \\\"\\\"\\\"Analyze FASTA sequence text or PDB coordinate text locally.\\\"\\\"\\\"\\n\\n    def analyze(self, content: str, input_format: str = \\\"auto\\\") -> dict[str, Any]:\\n        \\\"\\\"\\\"Return descriptive sequence or coordinate-file properties.\\n\\n        ``input_format`` may be ``auto``, ``fasta``, or ``pdb``. Input is\\n        processed locally and is never persisted by this class.\\n        \\\"\\\"\\\"\\n        if not isinstance(content, str) or not content.strip():\\n            raise ValueError(\\\"Input must be non-empty text.\\\")\\n        if len(content.encode(\\\"utf-8\\\")) > MAX_INPUT_BYTES:\\n            raise ValueError(f\\\"Input exceeds the {MAX_INPUT_BYTES}-byte limit.\\\")\\n\\n        selected = input_format.lower().strip()\\n        if selected == \\\"auto\\\":\\n            selected = _detect_format(content)\\n        if selected == \\\"fasta\\\":\\n            result = _analyze_fasta(content)\\n        elif selected == \\\"pdb\\\":\\n            result = _analyze_pdb(content)\\n        else:\\n            raise ValueError(\\\"input_format must be 'auto', 'fasta', or 'pdb'.\\\")\\n\\n        result[\\\"safety_notice\\\"] = (\\n            \\\"Descriptive research output only; not a diagnosis, clinical interpretation, \\\"\\n            \\\"or treatment recommendation.\\\"\\n        )\\n        return result\\n\\n\\ndef _detect_format(content: str) -> str:\\n    first_line = next((line.strip() for line in content.splitlines() if line.strip()), \\\"\\\")\\n    if first_line.startswith(\\\">\\\"):\\n        return \\\"fasta\\\"\\n    if any(line.startswith((\\\"ATOM  \\\", \\\"HETATM\\\")) for line in content.splitlines()):\\n        return \\\"pdb\\\"\\n    raise ValueError(\\\"Could not detect format. Specify 'fasta' or 'pdb'.\\\")\\n\\n\\ndef _analyze_fasta(content: str) -> dict[str, Any]:\\n    records: list[tuple[str, str]] = []\\n    identifier: str | None = None\\n    sequence_parts: list[str] = []\\n\\n    def finish_record() -> None:\\n        nonlocal identifier, sequence_parts\\n        if identifier is not None:\\n            sequence = \\\"\\\".join(sequence_parts).upper()\\n            if not sequence:\\n                raise ValueError(f\\\"FASTA record {identifier!r} has no sequence.\\\")\\n            records.append((identifier, sequence))\\n        sequence_parts = []\\n\\n    for line_number, raw_line in enumerate(content.splitlines(), start=1):\\n        line = raw_line.strip()\\n        if not line or line.startswith(\\\";\\\"):\\n            continue\\n        if line.startswith(\\\">\\\"):\\n            finish_record()\\n            identifier = line[1:].strip()\\n            if not identifier:\\n                raise ValueError(f\\\"FASTA header on line {line_number} is empty.\\\")\\n        else:\\n            if identifier is None:\\n                raise ValueError(\\\"FASTA sequence data must follow a '>' header.\\\")\\n            sequence_parts.append(\\\"\\\".join(line.split()))\\n    finish_record()\\n\\n    if not records:\\n        raise ValueError(\\\"No FASTA records were found.\\\")\\n    return {\\n        \\\"analysis_type\\\": \\\"sequence_descriptive\\\",\\n        \\\"record_count\\\": len(records),\\n        \\\"records\\\": [_summarize_sequence(name, sequence) for name, sequence in records],\\n    }\\n\\n\\ndef _summarize_sequence(identifier: str, sequence: str) -> dict[str, Any]:\\n    letters = set(sequence)\\n    if letters <= DNA_ALPHABET or letters <= RNA_ALPHABET:\\n        molecule_type = \\\"RNA\\\" if \\\"U\\\" in letters else (\\\"DNA\\\" if \\\"T\\\" in letters else \\\"nucleic_acid_unspecified\\\")\\n        counts = Counter(sequence)\\n        result: dict[str, Any] = {\\n            \\\"id\\\": identifier,\\n            \\\"molecule_type\\\": molecule_type,\\n            \\\"length\\\": len(sequence),\\n            \\\"composition\\\": dict(sorted(counts.items())),\\n        }\\n        canonical_alphabet = \\\"ACGU\\\" if molecule_type == \\\"RNA\\\" else \\\"ACGT\\\"\\n        canonical = sum(counts.get(base, 0) for base in canonical_alphabet)\\n        gc = counts.get(\\\"G\\\", 0) + counts.get(\\\"C\\\", 0)\\n        result[\\\"gc_percent_of_canonical_bases\\\"] = round(gc * 100 / canonical, 2) if canonical else None\\n        result[\\\"ambiguous_symbol_count\\\"] = len(sequence) - canonical\\n        return result\\n\\n    if letters <= PROTEIN_ALPHABET:\\n        return {\\n            \\\"id\\\": identifier,\\n            \\\"molecule_type\\\": \\\"protein\\\",\\n            \\\"length\\\": len(sequence),\\n            \\\"composition\\\": dict(sorted(Counter(sequence).items())),\\n            \\\"stop_symbol_count\\\": sequence.count(\\\"*\\\"),\\n        }\\n\\n    invalid = \\\"\\\".join(sorted(letters - DNA_ALPHABET - PROTEIN_ALPHABET))\\n    raise ValueError(f\\\"Sequence {identifier!r} contains unsupported symbols: {invalid!r}.\\\")\\n\\n\\ndef _analyze_pdb(content: str) -> dict[str, Any]:\\n    atoms = 0\\n    residues: set[tuple[str, str, str, str]] = set()\\n    chains: set[str] = set()\\n    coordinates: list[tuple[float, float, float]] = []\\n\\n    for line_number, line in enumerate(content.splitlines(), start=1):\\n        if not line.startswith((\\\"ATOM  \\\", \\\"HETATM\\\")):\\n            continue\\n        if len(line) < 54:\\n            raise ValueError(f\\\"PDB atom record on line {line_number} is too short.\\\")\\n        try:\\n            x, y, z = (float(line[start:end]) for start, end in ((30, 38), (38, 46), (46, 54)))\\n        except ValueError as exc:\\n            raise ValueError(f\\\"PDB coordinates on line {line_number} are invalid.\\\") from exc\\n        atoms += 1\\n        chains.add(line[21:22].strip() or \\\"_\\\")\\n        residues.add((line[21:22], line[22:26].strip(), line[26:27], line[17:20].strip()))\\n        coordinates.append((x, y, z))\\n\\n    if not atoms:\\n        raise ValueError(\\\"No ATOM or HETATM records were found in the PDB input.\\\")\\n    bounds = {\\n        axis: {\\n            \\\"min\\\": round(min(point[index] for point in coordinates), 3),\\n            \\\"max\\\": round(max(point[index] for point in coordinates), 3),\\n        }\\n        for index, axis in enumerate((\\\"x\\\", \\\"y\\\", \\\"z\\\"))\\n    }\\n    return {\\n        \\\"analysis_type\\\": \\\"structure_descriptive\\\",\\n        \\\"format\\\": \\\"pdb\\\",\\n        \\\"atom_count\\\": atoms,\\n        \\\"residue_count\\\": len(residues),\\n        \\\"chain_ids\\\": sorted(chains),\\n        \\\"coordinate_bounds_angstrom\\\": bounds,\\n    }\\n\\n\\n__all__ = [\\\"GeneCrispyAgent\\\", \\\"MAX_INPUT_BYTES\\\"]\\n\",\"genecrispy/education.py\":\"\\\"\\\"\\\"Independent study support mapped to public UIC Phase 1 objectives.\\n\\nMappings are GENE CRISpy's interpretation, not UIC-authored teaching content.\\nNo network calls, patient assessment, genetic testing recommendations or AI.\\n\\\"\\\"\\\"\\nfrom copy import deepcopy\\n\\nSOURCE_URL = \\\"https://medicine.uic.edu/education/md/md-curriculum/phase-1-curriculum/\\\"\\nSOURCE_REVIEWED = \\\"2026-11-14\\\"\\nNOTICE = (\\n    \\\"Independent GENE CRISpy study aid; not affiliated with or endorsed by UIC. \\\"\\n    \\\"Public-objective mapping only, not a complete syllabus or clinical interpretation.\\\"\\n)\\n_LESSONS = [\\n    {\\n        \\\"id\\\": \\\"sequence_foundations\\\",\\n        \\\"title\\\": \\\"DNA, RNA and base composition\\\",\\n        \\\"objective_ids\\\": [\\\"B1.4\\\"],\\n        \\\"mapping\\\": \\\"Preparatory foundation for genetics; sequence composition is not a disease-risk measure.\\\",\\n        \\\"explanation\\\": (\\n            \\\"DNA uses A, C, G and T; RNA uses A, C, G and U. \\\"\\n            \\\"GC percentage describes the fraction of canonical bases that are G or C. \\\"\\n            \\\"A short sequence containing neither T nor U may not distinguish DNA from RNA. \\\"\\n            \\\"Sequence alphabets can also overlap with protein alphabets.\\\"\\n        ),\\n        \\\"reflection\\\": \\\"Why can GCGCGC have 100% GC without establishing whether it is DNA or RNA?\\\",\\n        \\\"answer\\\": \\\"G and C occur in both DNA and RNA; molecule identity requires additional context.\\\",\\n    },\\n    {\\n        \\\"id\\\": \\\"inheritance\\\",\\n        \\\"title\\\": \\\"Inheritance and family history\\\",\\n        \\\"objective_ids\\\": [\\\"B1.4\\\"],\\n        \\\"mapping\\\": \\\"Study inheritance principles using a deliberately simplified example.\\\",\\n        \\\"explanation\\\": (\\n            \\\"In a simple autosomal recessive Aa by Aa model, each parent contributes \\\"\\n            \\\"A or a with equal probability. Offspring genotype probabilities are \\\"\\n            \\\"AA: 25%, Aa: 50%, aa: 25%. This assumes Mendelian segregation; \\\"\\n            \\\"phenotype also depends on the model's penetrance and other assumptions. \\\"\\n            \\\"This is not a calculation for an actual family.\\\"\\n        ),\\n        \\\"reflection\\\": \\\"Does one child's genotype change the probabilities for the next independent conception?\\\",\\n        \\\"answer\\\": \\\"No. Under these assumptions the genotype probabilities are the same for each conception.\\\",\\n    },\\n    {\\n        \\\"id\\\": \\\"testing_limits\\\",\\n        \\\"title\\\": \\\"Genetic testing and evidence limits\\\",\\n        \\\"objective_ids\\\": [\\\"B1.4\\\", \\\"B1.11\\\"],\\n        \\\"mapping\\\": \\\"Discuss what additional evidence a testing question needs; do not select a test for a patient.\\\",\\n        \\\"explanation\\\": (\\n            \\\"A FASTA summary does not identify a gene or a pathogenic variant. \\\"\\n            \\\"Interpretation needs context such as a reference sequence, genomic location, \\\"\\n            \\\"validated methods and appropriate evidence. Clinical assessment also needs \\\"\\n            \\\"personal and family history, consent and professional review.\\\"\\n        ),\\n        \\\"reflection\\\": \\\"Can matching GC percentages establish that two sequences have the same function?\\\",\\n        \\\"answer\\\": \\\"No. Matching composition does not establish sequence identity, function or clinical significance.\\\",\\n    },\\n    {\\n        \\\"id\\\": \\\"pharmacogenomics\\\",\\n        \\\"title\\\": \\\"Pharmacogenomics as a learning connection\\\",\\n        \\\"objective_ids\\\": [\\\"B1.8\\\"],\\n        \\\"mapping\\\": \\\"Conceptual connection only; no drug or dose recommendations.\\\",\\n        \\\"explanation\\\": (\\n            \\\"Inherited variation can influence drug handling or response. \\\"\\n            \\\"A descriptive sequence summary cannot establish a pharmacogenomic phenotype \\\"\\n            \\\"or determine a medication or dose.\\\"\\n        ),\\n        \\\"reflection\\\": \\\"What is missing between counting bases and predicting a drug response?\\\",\\n        \\\"answer\\\": \\\"Validated variant identification, relevant evidence and clinical context, among other requirements.\\\",\\n    },\\n    {\\n        \\\"id\\\": \\\"ethics\\\",\\n        \\\"title\\\": \\\"Genetics, consent and equitable access\\\",\\n        \\\"objective_ids\\\": [\\\"B1.13\\\"],\\n        \\\"mapping\\\": \\\"Reflect on ethical and structural factors in genetics education.\\\",\\n        \\\"explanation\\\": (\\n            \\\"Genetic information can affect relatives as well as the person tested. \\\"\\n            \\\"Educational workflows should consider consent, privacy, uncertainty and \\\"\\n            \\\"barriers to counseling or testing. These exercises use synthetic data.\\\"\\n        ),\\n        \\\"reflection\\\": \\\"Why should consent and privacy be considered before sharing genetic data?\\\",\\n        \\\"answer\\\": \\\"Genetic data can reveal sensitive information about individuals and biological relatives.\\\",\\n    },\\n]\\n\\n\\ndef get_m1_genetics_curriculum():\\n    \\\"\\\"\\\"Return a fresh, JSON-serializable curriculum snapshot.\\\"\\\"\\\"\\n    return {\\n        \\\"title\\\": \\\"GENE CRISpy M1 Genetics Foundations\\\",\\n        \\\"source_url\\\": SOURCE_URL,\\n        \\\"source_reviewed\\\": SOURCE_REVIEWED,\\n        \\\"notice\\\": NOTICE,\\n        \\\"coverage\\\": \\\"Selected Block 1 connections only; not full Phase 1 or M1 coverage.\\\",\\n        \\\"lessons\\\": deepcopy(_LESSONS),\\n    }\\n\\n\\ndef get_genetics_lesson(lesson_id):\\n    \\\"\\\"\\\"Return one lesson, raising ValueError for an unknown ID.\\\"\\\"\\\"\\n    for lesson in _LESSONS:\\n        if lesson[\\\"id\\\"] == lesson_id:\\n            return deepcopy(lesson)\\n    raise ValueError(f\\\"Unknown genetics lesson: {lesson_id!r}\\\")\\n\",\"tests/test_m1_education.py\":\"import unittest\\nfrom genecrispy import GeneCrispyAgent, get_m1_genetics_curriculum, get_genetics_lesson\\n\\n\\nclass M1EducationTests(unittest.TestCase):\\n    def test_snapshot_is_independent_and_source_traceable(self):\\n        curriculum = get_m1_genetics_curriculum()\\n        self.assertIn(\\\"medicine.uic.edu\\\", curriculum[\\\"source_url\\\"])\\n        self.assertIn(\\\"not affiliated\\\", curriculum[\\\"notice\\\"])\\n        curriculum[\\\"lessons\\\"][0][\\\"objective_ids\\\"].append(\\\"changed\\\")\\n        self.assertNotIn(\\\"changed\\\", get_m1_genetics_curriculum()[\\\"lessons\\\"][0][\\\"objective_ids\\\"])\\n\\n    def test_lesson_lookup_and_unknown_lesson(self):\\n        self.assertEqual(get_genetics_lesson(\\\"inheritance\\\")[\\\"objective_ids\\\"], [\\\"B1.4\\\"])\\n        with self.assertRaises(ValueError):\\n            get_genetics_lesson(\\\"missing\\\")\\n\\n    def test_rna_and_unresolved_nucleic_acid(self):\\n        agent = GeneCrispyAgent()\\n        rna = agent.analyze(\\\">rna\\\\nACGU\\\\n\\\")[\\\"records\\\"][0]\\n        self.assertEqual(rna[\\\"molecule_type\\\"], \\\"RNA\\\")\\n        self.assertEqual(rna[\\\"gc_percent_of_canonical_bases\\\"], 50.0)\\n        ambiguous = agent.analyze(\\\">unknown\\\\nGCGCGC\\\\n\\\")[\\\"records\\\"][0]\\n        self.assertEqual(ambiguous[\\\"molecule_type\\\"], \\\"nucleic_acid_unspecified\\\")\\n\\n    def test_original_samples_and_protein(self):\\n        agent = GeneCrispyAgent()\\n        records = agent.analyze(\\\">A\\\\nACGTGC\\\\n>B\\\\nATATAT\\\\n>C\\\\nGCGCGC\\\\n\\\")[\\\"records\\\"]\\n        self.assertEqual([r[\\\"gc_percent_of_canonical_bases\\\"] for r in records], [66.67, 0.0, 100.0])\\n        self.assertEqual(agent.analyze(\\\">protein\\\\nMPEPTIDE\\\\n\\\")[\\\"records\\\"][0][\\\"molecule_type\\\"], \\\"protein\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    unittest.main()\\n\"}")
for relative_path, content in updates.items():
    target = repo / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")

# Clear earlier imports so an older first-run checkout is not reused.
for name in list(sys.modules):
    if name == "genecrispy" or name.startswith("genecrispy."):
        del sys.modules[name]
sys.path.insert(0, str(repo))
from genecrispy import GeneCrispyAgent, __version__
from genecrispy import get_m1_genetics_curriculum, get_genetics_lesson

agent = GeneCrispyAgent()
print("GENE CRISpy", __version__)
print("Loaded from:", repo)
subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], check=True)


## 2. Read the learning map
The objective connections below are our study mapping, not additional official UIC objectives. B1.4 connects inheritance and genetic testing; B1.8 includes pharmacogenomics; B1.11 supports evidence appraisal; B1.13 includes ethical and structural context.


In [ ]:
curriculum = get_m1_genetics_curriculum()
print(curriculum["notice"])
print("Source:", curriculum["source_url"])
for lesson in curriculum["lessons"]:
    print(lesson["id"], "—", lesson["title"], "—", ", ".join(lesson["objective_ids"]))


## 3. Repeat your three DNA examples and add RNA
FASTA is a text format. GC percentage is a composition statistic, not a measure of disease risk. GCGCGC does not establish DNA versus RNA identity from letters alone.


In [ ]:
samples = ">sample_A\nACGTGC\n>sample_B\nATATAT\n>sample_C\nGCGCGC\n>sample_RNA\nACGU\n"
results = agent.analyze(samples, input_format="fasta")
for record in results["records"]:
    print(record["id"], record["molecule_type"], "length:", record["length"],
          "GC%:", record["gc_percent_of_canonical_bases"])
assert results["records"][3]["molecule_type"] == "RNA"


## 4. Learn and reflect
Read each explanation, write your own response, then compare with the model answer in the next cell. No learner responses are collected or sent anywhere.


In [ ]:
for lesson in curriculum["lessons"]:
    print("\n" + lesson["title"])
    print(lesson["explanation"])
    print("Reflect:", lesson["reflection"])


In [ ]:
for lesson in curriculum["lessons"]:
    print(lesson["id"] + ":", lesson["answer"])


## 5. Check the genetics learning path
These focused checks do not load the legacy PatientWorld/API modules. The pre-existing full test suite still has unrelated import blockers.


In [ ]:
subprocess.run([
    sys.executable, "-m", "unittest", "discover",
    "-s", "tests", "-p", "test_m1_education.py", "-v"
], cwd=repo, check=True)


## 6. Download your results (optional)
Downloads a CSV and JSON bundle. Save a copy of the notebook to your Google Drive as well; runtime files are temporary.


In [ ]:
import csv
import json
import zipfile
from google.colab import files

csv_path = Path("/content/GENE_CRISpy_M1_results.csv")
fields = ["id", "molecule_type", "length", "gc_percent_of_canonical_bases", "ambiguous_symbol_count"]
with csv_path.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(results["records"])
json_path = Path("/content/GENE_CRISpy_M1_learning.json")
json_path.write_text(json.dumps({"curriculum": curriculum, "results": results}, indent=2))
bundle = Path("/content/GENE_CRISpy_M1_results.zip")
with zipfile.ZipFile(bundle, "w") as archive:
    archive.write(csv_path, csv_path.name)
    archive.write(json_path, json_path.name)
files.download(str(bundle))
